# 映画のレコメンデーション: 検索（Retrieval）

**学習目標**

このノートブックでは、MovieLens データセットを使用して 2 タワー検索モデル（two-tower retrieval model）を構築し、トレーニングします。

具体的には以下を行います。

1. データを取得し、トレーニング セットとテスト セットに分割する。
2. 検索モデルを実装する。
3. モデルを適合（fit）させ、評価する。
4. 近似最近傍（ANN）インデックスを構築して、効率的なサービング用にモデルをエクスポートする。



## はじめに
現実世界のレコメンダー システムは、多くの場合 2 つのステージで構成されます。

1. **検索（Retrieval）ステージ**: すべての候補の中から、最初の数百個の候補を選択する役割を担います。このモデルの主な目的は、ユーザーが興味のないすべての候補を効率的に除外することです。検索モデルは何百万もの候補を扱う可能性があるため、計算効率が高くなければなりません。
2. **ランキング（Ranking）ステージ**: 検索モデルの出力を受け取り、それらを細かく調整して、可能な限り最適な一握りのレコメンデーションを選択します。その役割は、ユーザーが関心を持つ可能性のあるアイテムのセットを、有望な候補の短いリストに絞り込むことです。

このノートブックでは、最初のステージである検索に焦点を当てます。

検索モデルは多くの場合、2 つのサブモデルで構成されます。

1. クエリ特徴量を使用してクエリ表現（通常は固定次元の埋め込みベクトル）を計算する**クエリ モデル**。
2. 候補特徴量を使用して候補表現（同じサイズのベクトル）を計算する**候補モデル**。

次に、2 つのモデルの出力を掛け合わせてクエリと候補の親和性スコア（affinity score）を算出し、スコアが高いほど候補とクエリのマッチ度が高いことを示します。

この 2 つのサブモデルからなるアーキテクチャは、**2 タワー（two-tower）** モデルと呼ばれます。

### データセット

MovieLens データセットは、ミネソタ大学の [GroupLens](https://grouplens.org/datasets/movielens/) 研究グループが提供する古典的なデータセットです。複数のユーザーによって映画に付与された評価のセットが含まれており、レコメンダー システム研究の中心的なデータセットとなっています。

このデータは、次の 2 つの方法で扱うことができます。

1. ユーザーがどの映画を視聴（および評価）し、どの映画を視聴しなかったかを表していると解釈できます。これは**暗黙的フィードバック（implicit feedback）**の一種であり、ユーザーの視聴履歴から、どの作品を見たいと思い、どの作品を見たくないのかがわかります。
2. ユーザーが実際に視聴した映画をどの程度気に入ったかを表していると見ることもできます。これは**明示的フィードバック（explicit feedback）**の一種です。ユーザーが映画を視聴した場合、付与された評価を見ることで、どれだけ気に入ったかが大まかにわかります。

このノートブックでは、カタログの中からユーザーが視聴する可能性の高い映画のセットを予測するモデルである検索システムに焦点を当てます。検索システムでは暗黙的データの方が有用であることが多いため、ここでは MovieLens を暗黙的システムとして扱います。つまり、ユーザーが視聴したすべての映画が正例（positive example）となり、視聴していないすべての映画が暗黙的な負例（implicit negative example）となります。

各学習目標は、ノートブック内の __#TODO__ に対応しています。実行前にノートブックのセルのコードを完成させてください。参考として [解答例](https://github.com/GoogleCloudPlatform/training-data-analyst/blob/master/courses/machine_learning/deepdive2/recommendation_systems/solutions/basic_retrieval.ipynb) を参照してください。


## セットアップ

In [ ]:
# 必要なモジュールのインポート
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [3]:
import pprint
import tempfile
from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

In [4]:
import tensorflow_recommenders as tfrs

## データセットの準備

まずデータを確認してみましょう。

[TensorFlow Datasets](https://www.tensorflow.org/datasets) の MovieLens データセットを使用します。`movielens/100k_ratings` をロードすると評価データを含む `tf.data.Dataset` オブジェクトが取得でき、`movielens/100k_movies` をロードすると映画データのみを含む `tf.data.Dataset` オブジェクトが取得できます。

MovieLens データセットには事前に定義された分割（split）がないため、すべてのデータが `train` スプリットに含まれていることに注意してください。

In [5]:
# TODO 1 - ここにコードを記述してください。
# 評価データ。
# Features of all the available movies.

Shuffling and writing examples to /home/kbuilder/tensorflow_datasets/movielens/100k-ratings/0.1.0.incompleteN2R8B5/movielens-train.tfrecord
Dataset movielens downloaded and prepared to /home/kbuilder/tensorflow_datasets/movielens/100k-ratings/0.1.0. Subsequent calls will reuse this data.
Shuffling and writing examples to /home/kbuilder/tensorflow_datasets/movielens/100k-movies/0.1.0.incompleteH88HCK/movielens-train.tfrecord
Dataset movielens downloaded and prepared to /home/kbuilder/tensorflow_datasets/movielens/100k-movies/0.1.0. Subsequent calls will reuse this data.


評価データセットは、映画 ID、ユーザー ID、割り当てられた評価、タイムスタンプ、映画情報、ユーザー情報の辞書を返します。

In [6]:
# ユーザー情報と映画情報の表示
for x in ratings.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'bucketized_user_age': 45.0,
 'movie_genres': array([7]),
 'movie_id': b'357',
 'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'timestamp': 879024327,
 'user_gender': True,
 'user_id': b'138',
 'user_occupation_label': 4,
 'user_occupation_text': b'doctor',
 'user_rating': 4.0,
 'user_zip_code': b'53211'}


映画データセットには、映画 ID、映画タイトル、およびその映画が属するジャンルに関するデータが含まれています。ジャンルは整数のラベルでエンコードされていることに注意してください。

In [7]:
# 所属するジャンルに関するデータの表示
for x in movies.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'movie_genres': array([4]),
 'movie_id': b'1681',
 'movie_title': b'You So Crazy (1994)'}


この例では、評価データに焦点を当てます。他のノートブックでは、モデルの品質を向上させるために映画情報データも活用する方法を探求しています。

データセット内では `user_id` と `movie_title` フィールドのみを保持します。

In [8]:
# ここでは評価データに焦点を当てます
ratings = ratings.map(
    lambda x: {
        "movie_title": x["movie_title"],
        "user_id": x["user_id"],
    }
)
movies = movies.map(lambda x: x["movie_title"])

モデルの適合と評価を行うには、トレーニング セットと評価セットに分割する必要があります。実際のプロダクション向けレコメンダー システムでは、これは時間に基づいて行われる可能性が最も高くなります。つまり、時間 $T$ までのデータを使用して、$T$ 以降のインタラクションを予測します。


ただし、このシンプルな例ではランダム分割を使用し、評価の 80% をトレーニング セットに、20% をテスト セットに割り当てます。

In [9]:
# ここでは tf.random モジュールを使用して、最初の次元でテンソルをランダムにシャッフルします
SEED = 42

tf.random.set_seed(SEED)
shuffled = ratings.shuffle(100_000, seed=SEED, reshuffle_each_iteration=False)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

データ内に存在する一意のユーザー ID と映画タイトルも把握しておきましょう。

モデル内でカテゴリ特徴量の生数値を埋め込みベクトルにマッピングできるようにする必要があるため、これは重要です。そのためには、生の特徴量の値を連続した範囲の整数にマッピングする語彙（ボキャブラリ）が必要です。これにより、埋め込みテーブル内の対応する埋め込みを検索できるようになります。

In [10]:
# 埋め込みテーブルに従って対応するデータを表示。
movie_titles = movies.batch(1_000)
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:10]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)',
       b'2 Days in the Valley (1996)',
       b'20,000 Leagues Under the Sea (1954)',
       b'2001: A Space Odyssey (1968)',
       b'3 Ninjas: High Noon At Mega Mountain (1998)',
       b'39 Steps, The (1935)'], dtype=object)

## モデルの実装

モデルのアーキテクチャの選択は、レコメンダー モデリングの重要な部分です。

ここでは 2 タワー検索モデルを構築しているため、各タワーを個別に構築し、最終モデルでそれらを結合できます。

### クエリ タワー

まずクエリ タワーから始めましょう。

最初のステップは、クエリと候補の表現の次元数を決定することです。

In [11]:
embedding_dimension = 32

値を大きくすると、モデルの精度が向上する可能性がありますが、適合に時間がかかり、過学習（オーバーフィッティング）しやすくなります。

次はモデル自体です。ここでは、Keras の前処理レイヤーを使用して、まずユーザー ID を整数に変換し、次に `Embedding` レイヤーを介してそれらをユーザー埋め込みに変換します。語彙として、先ほど計算した一意のユーザー ID のリストを使用していることに注目してください。

In [12]:
user_model = tf.keras.Sequential(
    [
        tf.keras.layers.experimental.preprocessing.StringLookup(
            vocabulary=unique_user_ids, mask_token=None
        ),
        # 未知のトークンに対応するために追加の埋め込みを追加します。
        tf.keras.layers.Embedding(
            len(unique_user_ids) + 1, embedding_dimension
        ),
    ]
)

このような単純なモデルに対して `tf.keras.Model` のサブクラスを定義するのは大げさかもしれませんが、最終的に `embedding_dimension` の幅の出力を返す限り、標準の Keras コンポーネントを使用して任意の複雑なモデルに簡単に拡張できます。

### 候補タワー

候補タワーに対しても同様の処理を行うことができます。

In [13]:
movie_model = tf.keras.Sequential(
    [
        tf.keras.layers.experimental.preprocessing.StringLookup(
            vocabulary=unique_movie_titles, mask_token=None
        ),
        tf.keras.layers.Embedding(
            len(unique_movie_titles) + 1, embedding_dimension
        ),
    ]
)

### メトリクス

トレーニング データには正例の（ユーザー, 映画）ペア（実際の評価が存在するペア）があります。モデルの性能を把握するには、このペアに対してモデルが計算した親和性スコアを、他のすべての候補（すべての映画）のスコアと比較する必要があります。正例ペアの分類スコアが他のすべての候補よりも高い場合、モデルの精度が高いと判断できます。

これを行うには、`tfrs.metrics.FactorizedTopK` メトリクスを使用できます。このメトリクスには 1 つの必須引数があります。評価用の暗黙的な負例（明示的な評価を持たない映画候補）として使用される候補のデータセットです。
この例では、映画モデルを介して埋め込みに変換された `movies` データセットが該当します。

`FactorizedTopK` は複数のメトリクスを自動的に計算します。デフォルトでは上位 [1, 5, 10, 50, 100] の [`TopKCategoricalAccuracy`](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/TopKCategoricalAccuracy) です。<br>
Top-K 精度は、すべての負例候補（この場合はすべての映画）の中で、正例ペアが上位 K 位以内に正しくランク付けされた割合を表します。

In [14]:
# ここでは、tfrs.metrics.FactorizedTopK 関数が検索モデルによって提示された上位 K 個の候補全体のメトリクスを計算します。
metrics = tfrs.metrics.FactorizedTopK(
    candidates=movies.batch(128).map(movie_model)
)

### 損失

次のコンポーネントは、モデルのトレーニングに使用される損失です。TFRS には、これを簡単に行うためのいくつかの損失レイヤーとタスクが用意されています。

ここでは、損失関数とメトリクス計算を一つにまとめた便利なラッパーである `tfrs.task.Retrieval` タスク オブジェクトを使用します。

内部的には、`Retrieval` タスク レイヤーはクエリ埋め込みと候補埋め込みを乗算してスコアを計算し、損失（デフォルトではクロスエントロピー損失）を出力します。<br>
オプションで、メトリクスを指定してトレーニング中に計算することもできます。

In [15]:
# TODO 2 - ここにコードを記述してください。
# Here, the function bundles together the loss function and metric computation.

タスク自体は、クエリ埋め込みと候補埋め込みを引数として受け取り、計算された損失を返す Keras レイヤーです。これを使用して、モデルのトレーニング ループを実装します。

### モデル全体の構築

これらをすべて組み合わせて 1 つのモデルにまとめます。TFRS はモデル構築を効率化するベース モデル クラス（`tfrs.models.Model`）を提供しています。`__init__` メソッドでコンポーネントを設定し、生の特徴量を受け取って損失値を返す `compute_loss` メソッドを実装するだけで構築できます。

ベース モデルは、モデルを適合させるための適切なトレーニング ループの作成を自動的に処理します。

In [16]:
class MovielensModel(tfrs.Model):
    def __init__(self, user_model, movie_model):
        super().__init__()
        self.movie_model: tf.keras.Model = movie_model
        self.user_model: tf.keras.Model = user_model
        self.task: tf.keras.layers.Layer = task

    def compute_loss(self, features, training=False):
        # ユーザー特徴量を取り出してユーザー モデルに渡します。
        user_embeddings = self.user_model(features["user_id"])
        # そして映画の特徴量を取り出して映画モデルに渡し、
        # 埋め込みを取得します。
        positive_movie_embeddings = self.movie_model(features["movie_title"])

        # タスクが損失とメトリクスを計算します。
        return self.task(user_embeddings, positive_movie_embeddings)

`tfrs.Model` ベース クラスはシンプルで便利です。同じメソッドを使用してトレーニング損失とテスト損失の両方を計算できます。

内部的には、通常の Keras モデルと変わりません。以下のように `tf.keras.Model` を継承し、`train_step` 関数と `test_step` 関数をオーバーライドすることでも同じ機能を実現できます（詳細は [ガイド](https://www.tensorflow.org/guide/keras/customizing_what_happens_in_fit) を参照してください）。


```python
class MovielensModel(tf.keras.Model):
    def __init__(self, user_model, movie_model):
        super().__init__()
        self.movie_model: tf.keras.Model = movie_model
        self.user_model: tf.keras.Model = user_model
        self.task: tf.keras.layers.Layer = task

    def train_step(self, features):

        # Set up a gradient tape to record gradients.
        with tf.GradientTape() as tape:

            # Loss computation.
            user_embeddings = self.user_model(features["user_id"])
            positive_movie_embeddings = self.movie_model(
                features["movie_title"]
            )
            loss = self.task(user_embeddings, positive_movie_embeddings)

            # Handle regularization losses as well.
            regularization_loss = sum(self.losses)

            total_loss = loss + regularization_loss

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss

        return metrics

    def test_step(self, features):

        # Loss computation.
        user_embeddings = self.user_model(features["user_id"])
        positive_movie_embeddings = self.movie_model(features["movie_title"])
        loss = self.task(user_embeddings, positive_movie_embeddings)

        # Handle regularization losses as well.
        regularization_loss = sum(self.losses)

        total_loss = loss + regularization_loss

        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss

        return metrics
```



ただし、このノートブックではモデリングに集中しボイラープレート コードを抽象化するために、`tfrs.Model` ベース クラスを引き続き使用します。

## 適合（Fitting）と評価

モデルを定義した後は、標準の Keras の適合・評価ルーチンを使用してモデルのトレーニングと評価を行うことができます。

まず、モデルをインスタンス化しましょう。

In [18]:
# モデルのコンパイル。
model = MovielensModel(user_model, movie_model)
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

次に、トレーニング データと評価データをシャッフル、バッチ化、キャッシュします。

In [19]:
cached_train = train.shuffle(100_000).batch(8192).cache()
cached_test = test.batch(4096).cache()

次にモデルをトレーニングします。

In [20]:
# TODO 3a - ここにコードを記述してください。
# Training the model.

Epoch 1/3


10/10 [==============================] - 2s 238ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0011 - factorized_top_k/top_5_categorical_accuracy: 0.0090 - factorized_top_k/top_10_categorical_accuracy: 0.0190 - factorized_top_k/top_50_categorical_accuracy: 0.0961 - factorized_top_k/top_100_categorical_accuracy: 0.1735 - loss: 69885.1072 - regularization_loss: 0.0000e+00 - total_loss: 69885.1072
Epoch 2/3
10/10 [==============================] - 2s 224ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0028 - factorized_top_k/top_5_categorical_accuracy: 0.0187 - factorized_top_k/top_10_categorical_accuracy: 0.0374 - factorized_top_k/top_50_categorical_accuracy: 0.1686 - factorized_top_k/top_100_categorical_accuracy: 0.2919 - loss: 67523.3707 - regularization_loss: 0.0000e+00 - total_loss: 67523.3707
Epoch 3/3
10/10 [==============================] - 2s 219ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0035 - factorized_top_k/top_5_categorical_accuracy: 0.0222 - f

モデルのトレーニングが進むにつれて損失は減少し、一連の Top-K 検索メトリクスが更新されます。これらは、全候補セットから検索された上位 K 個のアイテムの中に真の正例が含まれているかどうかを示します。たとえば、Top-5 カテゴリ精度のメトリクスが 0.2 であれば、平均して 20% の確率で上位 5 件の検索アイテムに真の正例が含まれていることを示します。

なお、この例では評価時だけでなくトレーニング中にもメトリクスを評価しています。候補セットが大きい場合、これは非常に遅くなる可能性があるため、トレーニング中のメトリクス計算をオフにして評価時のみ実行するのが賢明な場合もあります。

最後に、テスト セットでモデルを評価します。

In [21]:
# TODO 3b - ここにコードを記述してください。
# Evaluating the model.

5/5 [==============================] - 1s 118ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0010 - factorized_top_k/top_5_categorical_accuracy: 0.0097 - factorized_top_k/top_10_categorical_accuracy: 0.0226 - factorized_top_k/top_50_categorical_accuracy: 0.1244 - factorized_top_k/top_100_categorical_accuracy: 0.2327 - loss: 31079.0628 - regularization_loss: 0.0000e+00 - total_loss: 31079.0628


{'factorized_top_k/top_1_categorical_accuracy': 0.0010000000474974513,
 'factorized_top_k/top_5_categorical_accuracy': 0.009650000371038914,
 'factorized_top_k/top_10_categorical_accuracy': 0.022600000724196434,
 'factorized_top_k/top_50_categorical_accuracy': 0.12439999729394913,
 'factorized_top_k/top_100_categorical_accuracy': 0.23270000517368317,
 'loss': 28244.76953125,
 'regularization_loss': 0,
 'total_loss': 28244.76953125}

テスト セットでのパフォーマンスは、トレーニング時のパフォーマンスよりも大幅に低くなっています。これには 2 つの要因があります。

1. モデルは、単純に暗記できるため、すでに見たデータに対して優れたパフォーマンスを発揮する傾向があります。この過学習（オーバーフィッティング）現象は、モデルのパラメータ数が多い場合に特に顕著になります。これは、モデルの正則化や、未知のデータに対してより汎化しやすくするユーザー特徴量や映画特徴量の活用によって緩和できます。
2. モデルが、ユーザーがすでに視聴した映画の一部を再度レコメンドしていることです。これらの既知の正例視聴が、テスト対象の映画を上位 K 件のレコメンデーションから締め出してしまう可能性があります。

2 つ目の現象は、テスト用のレコメンデーションから過去に視聴した映画を除外することで対処できます。このアプローチはレコメンダー システムの文献では比較的一般的ですが、このノートブックでは採用していません。過去に視聴した作品をレコメンドしないことが重要である場合、適切に指定されたモデルが過去のユーザー履歴やコンテキスト情報からこの動作を自動的に学習することが期待されます。さらに、同じアイテムを複数回レコメンドすることが適切な場合も多々あります（たとえば、定番の人気 TV シリーズや定期的に購入される商品など）。

## 予測の実行

モデルが完成したので、予測を実行できるようにします。これには `tfrs.layers.factorized_top_k.BruteForce` レイヤーを使用できます。

In [22]:
# 生のクエリ特徴量を受け取るモデルを作成し、
index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
# 映画データセット全体から映画をレコメンドします。
index.index_from_dataset(
    tf.data.Dataset.zip(
        (movies.batch(100), movies.batch(100).map(model.movie_model))
    )
)

# レコメンデーションの取得。
_, titles = index(tf.constant(["42"]))
print(f"Recommendations for user 42: {titles[0, :3]}")

Recommendations for user 42: [b'Bridges of Madison County, The (1995)'
 b'Father of the Bride Part II (1995)' b'Rudy (1993)']


もちろん、候補となるアイテムが多数存在するモデルをサービングする場合、`BruteForce` レイヤーは遅すぎます。以降のセクションでは、近似検索インデックスを使用してこれを高速化する方法を示します。

## モデル サービング

モデルをトレーニングした後は、それをデプロイする方法が必要です。

2 タワー検索モデルにおいて、サービングには 2 つのコンポーネントがあります。

- **サービング クエリ モデル**: クエリの特徴量を受け取り、それらをクエリ埋め込みに変換します。
- **サービング 候補モデル**: 多くの場合、クエリ モデルによって生成されたクエリに応じて候補を迅速に近似検索できる、近似最近傍（ANN）インデックスの形式をとります。

TFRS では、両方のコンポーネントを 1 つのエクスポート可能なモデルにパッケージ化できるため、生のユーザー ID を受け取ってそのユーザー向けの上位映画タイトルを返すモデルが得られます。これは、モデルを `SavedModel` 形式にエクスポートすることで行われ、[TensorFlow Serving](https://www.tensorflow.org/tfx/guide/serving) を使用したサービングが可能になります。

このようなモデルをデプロイするには、上で作成した `BruteForce` レイヤーをエクスポートするだけです。

In [23]:
# クエリ モデルのエクスポート。
# TODO 4 - ここにコードを記述してください。

  # インデックスの保存。
  tf.saved_model.save(index, path)

  # 再ロード（TensorFlow Serving でも実行可能）。
  loaded = tf.saved_model.load(path)

  # ユーザー ID を渡して、予測された上位の映画タイトルを取得。
  scores, titles = loaded(["42"])

  print(f"Recommendations: {titles[0][:3]}")

2021-09-29 13:25:05.366590: W tensorflow/python/util/util.cc:348] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.


INFO:tensorflow:Assets written to: /tmp/tmpklbnqnat/model/assets


INFO:tensorflow:Assets written to: /tmp/tmpklbnqnat/model/assets


Recommendations: [b'Bridges of Madison County, The (1995)'
 b'Father of the Bride Part II (1995)' b'Rudy (1993)']


近似検索インデックスをエクスポートして予測を高速化することもできます。これにより、数千万の候補のセットから効率的にレコメンデーションを提示することが可能になります。

高速な近似検索モデルの使用とチューニングの詳細については、[効率的なサービング（efficient serving）](https://tensorflow.org/recommenders/examples/efficient_serving) ノートブックを参照してください。

## 次のステップ

これで検索ノートブックは終了です。

ここで紹介した内容を発展させるには、以下を参照してください。

1. マルチタスク モデルの学習: 評価とクリックの共同最適化。
2. 映画メタデータの活用: コールド スタートを緩和するための、より複雑な映画モデルの構築。